# Notebook 01.2 — Aquisição da infraestrutura viária e edificações (Overture Maps)

**Projeto:** Acessibilidade Geográfica às UBS de Teresina: roteiro computacional AE2SFCA  
**Programa:** MAPEPROF - Mestrado Profissional em Planejamento Urbano e Regional / IFPI  
**Autor:** Felipe Ramos Dantas  
**Orientador:** Prof. Dr. Antonio Joaquim da Silva  
**Coorientador:** Prof. Dr. Reurysson Chagas de Sousa Morais  
**Repositório:** https://github.com/felipedantas-pi/ae2sfca-ubs  
**Última atualização:** 2026-06-10

## Objetivo

Utilizar a área de estudo pré-processada no **Notebook 01.1** (Zona Urbana + *buffer* de 5 km) para extrair da **Overture Maps Foundation** os dados de infraestrutura urbana de Teresina: malha viária (*segment* e *connector*), pegadas de construção (*building*) e, ao final, os estabelecimentos de saúde do CNES.

## Saídas

Gravadas em `dados/externos/overture/`:

| Arquivo | Consumido por | Descrição |
|---|---|---|
| `zonaUrbanaReal_5km_segmentos.geojson` / `.parquet` | NB 02.1 | Segmentos viários brutos (arestas do grafo) |
| `zonaUrbanaReal_5km_conectores.geojson` / `.parquet` | NB 02.1 | Conectores viários brutos (nós do grafo) |
| `teresina_building_utm.geojson` | NB 01.3 / NB 05 | Pegadas de construção (proxy espacial de domicílios) |

Gravada em `dados/externos/cnes/`:

| Arquivo | Consumido por | Descrição |
|---|---|---|
| `cnes_pysus_pi.csv` | NB 04/05 | Estabelecimentos de saúde do Piauí (CNES/DataSUS) |

## Pré-requisitos

- **NB 01.1** executado (recortes territoriais em `dados/externos/ibge/`).
- Conexão de internet estável (download da Overture pode levar minutos).

---

In [1]:
# ── 1. IMPORTAÇÕES E CONFIGURAÇÃO GLOBAL ────────────────────────────────────
import geopandas as gpd
import city2graph as c2g
from overturemaps import geodataframe as overture_gdf

# Caminhos e CRS centralizados (ver src/mapeprof/config.py)
from mapeprof.config import (
    EXT_IBGE,        # dados/externos/ibge     — recortes do NB 01.1
    EXT_OVERTURE,    # dados/externos/overture — saídas deste notebook
    CRS_METRICO,     # EPSG:31983 — SIRGAS 2000 / UTM 23S
    CRS_WGS84,       # EPSG:4326  — exigido pela API da Overture
    criar_diretorios,
)

criar_diretorios()
print("Dependências e diretórios prontos.")
print(f"city2graph: {c2g.__version__ if hasattr(c2g, '__version__') else 'development'}")

Dependências e diretórios prontos.
city2graph: 0.3.1


In [2]:
# ── 2. IMPORTAÇÃO DOS DADOS DO NOTEBOOK 01.1 ────────────────────────────────
print("🗺️ Carregando recortes espaciais gerados no Notebook 01.1...")

# Polígonos em UTM (EPSG:31983), gravados pelo NB 01.1
gdf_zonaUrbana   = gpd.read_parquet(EXT_IBGE / "teresina_zonaUrbana_utm.parquet")
gdf_zonaUrbanab5kms = gpd.read_parquet(EXT_IBGE / "teresina_zonaUrbana_buffer5kClip_utm.parquet")

# A API da Overture só aceita WGS 84 — reprojeta antes da requisição
gdf_zonaUrbana_wgs   = gdf_zonaUrbana.to_crs(CRS_WGS84)
gdf_zonaUrbana_buffer5km_wgs = gdf_zonaUrbanab5kms.to_crs(CRS_WGS84)

print("✅ Recortes carregados e reprojetados para WGS 84.")

🗺️ Carregando recortes espaciais gerados no Notebook 01.1...
✅ Recortes carregados e reprojetados para WGS 84.


In [5]:
# ── 3. DOWNLOAD SEGMENTOS E CONECTORES ──────────────────────────────────────
subdatasets = ["segment", "connector"]
RELEASE_OVERTURE = '2026-06-17.0'   # fixa a versão p/ reprodutibilidade da dissertação

In [6]:
print(f"🛣️ Extraindo malha viária (release {RELEASE_OVERTURE})...")

dados_viarios = c2g.load_overture_data(
    area=gdf_zonaUrbana_buffer5km_wgs,       # área de estudo (ZU + buffer 5 km)
    types=subdatasets,
    output_dir=EXT_OVERTURE,
    prefix='zonaUrbana_5km_',
    save_to_file=False,            # mantém em memória para reprojetar antes de salvar
    return_data=True,
    release=RELEASE_OVERTURE,
    use_stac=False,
)

print("🔄 Reprojetando para UTM e salvando...")
segments_metric   = dados_viarios['segment'].to_crs(CRS_METRICO)
connectors_metric = dados_viarios['connector'].to_crs(CRS_METRICO)

# GeoJSON: formato seguro p/ as listas aninhadas da Overture (consumido pelo NB 02.1)
segments_metric.to_file(EXT_OVERTURE / "zonaUrbana_5km_segmentos.geojson", driver="GeoJSON")
connectors_metric.to_file(EXT_OVERTURE / "zonaUrbana_5km_conectores.geojson", driver="GeoJSON")

# Parquet: leitura rápida para análises locais
segments_metric.to_parquet(EXT_OVERTURE / "zonaUrbana_5km_segmentos.parquet", index=False)
connectors_metric.to_parquet(EXT_OVERTURE / "zonaUrbana_5km_conectores.parquet", index=False)

print(f"✅ {len(segments_metric):,} segmentos e {len(connectors_metric):,} conectores salvos.")

🛣️ Extraindo malha viária (release 2026-06-17.0)...
🔄 Reprojetando para UTM e salvando...
✅ 46,970 segmentos e 35,183 conectores salvos.


In [7]:
# ── 4. DOWNLOAD DAS PEGADAS DE CONSTRUÇÃO ────────────────────────────────────

print("🏠 Calculando limites do município para extração de edificações...")

# Extrai o retângulo envolvente (Bounding Box) automático da zona urbana de Teresina [minX, minY, maxX, maxY]
bounds = gdf_zonaUrbana_wgs.total_bounds
bbox_zonaUrbana = (bounds[0], bounds[1], bounds[2], bounds[3])

print(f"📡 Solicitando base de 'buildings' para a Bounding Box: {bbox_zonaUrbana}")
# Chamada direta e segura à biblioteca nativa overturemaps
gdf_building = overture_gdf("building", bbox=bbox_zonaUrbana, release=RELEASE_OVERTURE)

# Confirma o CRS geográfico inicial
gdf_building.set_crs("EPSG:4326", inplace=True)

print(f"✅ Download concluído. Total de edificações mapeadas: {len(gdf_building):,}")

# mas mantemos o .geojson original caso exija interoperabilidade externa imediata)
print("💾 Reprojetando para UTM e exportando arquivo...")
caminho_buildings = EXT_OVERTURE / "zonaUrbana_building_utm.geojson"
gdf_building.to_crs(CRS_METRICO).to_file(caminho_buildings, driver="GeoJSON")
print(f"💾 Salvo em: {caminho_buildings}")

🏠 Calculando limites do município para extração de edificações...
📡 Solicitando base de 'buildings' para a Bounding Box: (np.float64(-42.86320059999998), np.float64(-5.229813399999962), np.float64(-42.66596079999994), np.float64(-4.956783199999967))
✅ Download concluído. Total de edificações mapeadas: 432,682
💾 Reprojetando para UTM e exportando arquivo...
💾 Salvo em: C:\Users\felipe\workspace_pcksa\ae2sfca_ubs\dados\externos\overture\zonaUrbana_building_utm.geojson
